# Job Market & Skills Analytics - Colab Exploratory Analysis

## Environment setup

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly",
    "openpyxl": "openpyxl",
    "pyarrow": "pyarrow",
}
missing = [package for module, package in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import html
import json
import math
import re
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab" if importlib.util.find_spec("google.colab") else "notebook_connected"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

IN_COLAB = importlib.util.find_spec("google.colab") is not None
DATA_DIR = Path("/content") if IN_COLAB else Path("dataset")
OUTPUT_DIR = Path("/content/job_market_outputs") if IN_COLAB else Path("data/processed/notebook")

print("Running in Colab:", IN_COLAB)
print("Dataset directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Running in Colab: True
Dataset directory: /content
Output directory: /content/job_market_outputs


## Dataset discovery

In [2]:
SUPPORTED_SUFFIXES = {".csv", ".tsv", ".xlsx", ".xlsm", ".json", ".jsonl", ".parquet"}

def discover_files(data_dir=DATA_DIR):
    return sorted(
        path for path in data_dir.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )

dataset_files = discover_files()
if not dataset_files and IN_COLAB:
    from google.colab import files
    print("No supported dataset found in /content. Select the dataset file to upload.")
    files.upload()
    dataset_files = discover_files()

if not dataset_files:
    raise FileNotFoundError(f"No CSV, TSV, Excel, JSON, or Parquet files found in {DATA_DIR.resolve()}")

inventory = pd.DataFrame([
    {"file": path.name, "type": path.suffix.lower(), "size_mb": path.stat().st_size / 1_000_000}
    for path in dataset_files
])
display(inventory.style.format({"size_mb": "{:.2f}"}))

,file,type,size_mb
0,indian-job-market-dataset-2025.xlsx,.xlsx,31.71


## Load and inspect every table


In [3]:
def load_tables(path):
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xlsm"}:
        workbook = pd.ExcelFile(path)
        return {sheet: pd.read_excel(path, sheet_name=sheet) for sheet in workbook.sheet_names}
    if suffix == ".csv":
        return {path.stem: pd.read_csv(path, low_memory=False)}
    if suffix == ".tsv":
        return {path.stem: pd.read_csv(path, sep="\t", low_memory=False)}
    if suffix in {".json", ".jsonl"}:
        return {path.stem: pd.read_json(path, lines=suffix == ".jsonl")}
    if suffix == ".parquet":
        return {path.stem: pd.read_parquet(path)}
    return {}

tables = {}
for path in dataset_files:
    for table_name, frame in load_tables(path).items():
        tables[(path.name, table_name)] = frame

table_summary = pd.DataFrame([
    {"file": file_name, "table": table_name, "rows": len(frame), "columns": frame.shape[1]}
    for (file_name, table_name), frame in tables.items()
]).sort_values(["rows", "columns"], ascending=False)
display(table_summary)

EXPECTED_FIELDS = {
    "title", "jobtitle", "jobid", "companyname", "company", "location",
    "salary", "experience", "tagsandskills", "skills", "jobdescription",
}

def schema_score(frame):
    compact = {re.sub(r"[^a-z0-9]", "", str(column).lower()) for column in frame.columns}
    return len(compact & EXPECTED_FIELDS)

candidates = [
    (schema_score(frame), len(frame), file_name, table_name, frame)
    for (file_name, table_name), frame in tables.items()
    if schema_score(frame) >= 2
]
if not candidates:
    raise ValueError("No job-listing table could be identified from the uploaded files.")

_, _, source_file, source_table, raw = max(candidates, key=lambda item: (item[0], item[1]))
print(f"Selected source: {source_file} — {source_table}")
print("Shape:", raw.shape)
display(raw.head())

,file,table,rows,columns
0,indian-job-market-dataset-2025.xlsx,Sheet1,97929,17


Selected source: indian-job-market-dataset-2025.xlsx — Sheet1
Shape: (97929, 17)


,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Power,Hiring,Recruitment,SR,Communication skills",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000.0,400000.0,2.0,4.0
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire Safety,Fire Engineering,Fire Prevention,Safety Management,Fire Manage...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162.0,4.0,"Ensure active Fire Protection System,such as Fire Hydrant system,Fire extinguishers,Fire Alarm smoke detector system...",300000.0,500000.0,6.0,11.0
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth marketing,Paid Marketing,Acquisition,Performance,Usage,Marketing",12-18 Yrs,Not disclosed,Chennai,1324750,2892.0,4.2,MBA Marketing (preferred Tier II or III B- School). <br><br>Experience <br><br> 12+ years exp in Digital Marketing w...,0.0,0.0,12.0,18.0
3,Medical Billing Executive,270925007443,INR,6 Days Ago,GNR Global Services,"Fluent English,Spoken English,Good English Communication,Medical,Medical billing,Billing,Fluent,English",0-3 Yrs,"70,000-2 Lacs PA","Mohali, Chandigarh, Kharar, Zirakpur",123804403,NaN,NaN,Job Title-Medical Billing Executive\nLocation-Mohali\nSalary-20-22k ctc\n\nBenefits:\nCab facility\nIncentives\nOne ...,70000.0,200000.0,0.0,3.0
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephrology,Group Product Management,Brand Marketing,Neurology,Brand Management",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134.0,3.4,Principal Tasks & Responsibilities : (Please write all the major jobs that the employee is required to carry out )<b...,800000.0,1800000.0,5.0,10.0


## Schema, missing values, duplicates, and samples


In [4]:
schema = pd.DataFrame({
    "column": raw.columns,
    "dtype": raw.dtypes.astype(str).values,
    "missing": raw.isna().sum().values,
    "missing_pct": raw.isna().mean().values,
    "unique_values": raw.nunique(dropna=True).values,
})
display(schema.style.format({"missing_pct": "{:.1%}"}))

compact_columns = {re.sub(r"[^a-z0-9]", "", str(column).lower()): column for column in raw.columns}
job_id_column = compact_columns.get("jobid")
print("Exact duplicate rows:", int(raw.duplicated().sum()))
if job_id_column:
    print("Duplicate job IDs beyond the first:", int(raw[job_id_column].duplicated().sum()))
display(raw.sample(min(5, len(raw)), random_state=42))

,column,dtype,missing,missing_pct,unique_values
0,title,object,0,0.0%,55104
1,jobId,int64,0,0.0%,97679
2,currency,object,0,0.0%,2
3,jobUploaded,object,0,0.0%,30
4,companyName,object,4,0.0%,18668
5,tagsAndSkills,object,571,0.6%,84307
6,experience,object,2105,2.1%,287
7,salary,object,0,0.0%,1339
8,location,object,0,0.0%,10066
9,companyId,int64,0,0.0%,18328


Exact duplicate rows: 247
Duplicate job IDs beyond the first: 250


,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
84176,Zonal Manager - Affordable Housing,260925918114,INR,7 Days Ago,Bajaj Finance,"Sales,Databases,Development,Talent Management,Data,Management,Due Diligence,Operations",14-15 Yrs,Not disclosed,Pune,122,8015.0,3.9,"Required Qualifications and Experience <br><br>Work Experience<br><br>Relevant Experience of 14-15 Years,out of whic...",0.0,0.0,14.0,15.0
39838,Management trainee collections,280825011954,INR,8 Days Ago,Genpact,"joining formalities,accounts receivable,orientation,accounts payable,hr generalist activities,sap,employee relations...",3-7 Yrs,Not disclosed,Jaipur,30975,38095.0,3.7,"<p><strong>Ready to shape the future of work? </strong></p><p><strong>\n</strong>At Genpact, we don’t just adapt to ...",0.0,0.0,3.0,7.0
77820,Senior C # Developer,300925029858,INR,3 Days Ago,Katapie Consulting Services,"C Hash,Motion Control,Opencv,Image Processing,Python,Messaging Systems,GUI,Motion",6-9 Yrs,20-30 Lacs PA,Remote,125133963,NaN,NaN,"Requires expertise in C# & Python , handling motion control, image processing, and hardware. Must manage actor-based...",2000000.0,3000000.0,6.0,9.0
68128,Machine Shop Electrical Maintenance,240925506188,INR,9 Days Ago,Happy Forging,"Electrical drawing,Air compressor,Basic,PLC,Programming,Siemens,Machine shop,Electrical maintenance",4-7 Yrs,Not disclosed,Ludhiana,124534690,136.0,3.7,"<li> <span> <i> </i> </span> <span> Troubleshooting on HMC, VMC, CNC (Fanuc/ Siemens/ Rexroth) </span> </li> <li> <s...",0.0,0.0,4.0,7.0
63223,Store Manager,300925030778,INR,2 Days Ago,Suvidha Placements,"Stores,Store Operations,consumable,Inventory Management,Steel and iron plant,Steel,Plant,Store management",12-16 Yrs,5-7 Lacs PA,"Barjora, Durgapur",85003,NaN,NaN,Managing stores of consumable and store items of steel company\nUnder stand the need of plan and mange the inventory...,500000.0,700000.0,12.0,16.0


## Reusable cleaning functions

In [5]:
MISSING_TEXT = {"", "nan", "none", "null", "n/a", "na", "not available"}

def normalize_column_name(name):
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(name).strip())
    return re.sub(r"[^a-zA-Z0-9]+", "_", text).strip("_").lower()

def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return pd.NA if text.casefold() in MISSING_TEXT else text

def clean_html_text(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"<\s*br\s*/?>|</p>|</li>", " ", str(value), flags=re.I)
    return clean_text(html.unescape(re.sub(r"<[^>]+>", " ", text)))

def salary_number(token, unit=None):
    number = float(token.replace(",", ""))
    unit = (unit or "").casefold()
    if unit.startswith(("lac", "lakh")) or unit == "lpa":
        number *= 100_000
    elif unit.startswith(("cr", "crore")):
        number *= 10_000_000
    elif unit in {"k", "thousand"}:
        number *= 1_000
    return number

def parse_salary(value):
    if pd.isna(value):
        return np.nan, np.nan
    text = str(value).strip().lower()
    if any(term in text for term in ("not disclosed", "unpaid", "negotiable", "not specified")):
        return np.nan, np.nan
    matches = re.findall(
        r"(?<![a-z])(\d[\d,]*(?:\.\d+)?)\s*(lacs?|lakhs?|lpa|crores?|cr|thousand|k)?",
        text,
    )[:2]
    if not matches:
        return np.nan, np.nan
    fallback_unit = next((unit for _, unit in reversed(matches) if unit), None)
    numbers = []
    for token, unit in matches:
        inferred = unit or (fallback_unit if "," not in token and float(token) < 1_000 else None)
        numbers.append(salary_number(token, inferred))
    minimum, maximum = (numbers[0], numbers[0]) if len(numbers) == 1 else sorted(numbers[:2])
    if "/month" in text or "per month" in text or re.search(r"\bpm\b", text):
        minimum, maximum = minimum * 12, maximum * 12
    return (np.nan, np.nan) if maximum <= 0 else (float(minimum), float(maximum))

def parse_experience(value):
    if pd.isna(value):
        return np.nan, np.nan
    text = str(value).strip().lower()
    if "fresher" in text or "no experience" in text:
        return 0.0, 0.0
    values = [float(item) for item in re.findall(r"\d+(?:\.\d+)?", text)]
    if not values:
        return np.nan, np.nan
    if len(values) == 1:
        return values[0], np.nan if "+" in text else values[0]
    return min(values[:2]), max(values[:2])

def experience_category(minimum, maximum):
    if pd.isna(minimum):
        return "Not specified"
    if minimum == 0 and pd.notna(maximum) and maximum <= 2:
        return "Fresher"
    if minimum <= 2:
        return "Entry Level"
    if minimum <= 5:
        return "Mid Level"
    return "Senior"

TITLE_RULES = [
    (r"\bdata\s+analyst\b|\banalyst.*data\b", "Data Analyst"),
    (r"\bbusiness\s+analyst\b", "Business Analyst"),
    (r"\b(?:bi|business intelligence)\s+analyst\b", "BI Analyst"),
    (r"\bdata\s+scientist\b", "Data Scientist"),
    (r"\bdata\s+engineer\b|\betl\s+(?:engineer|developer)\b", "Data Engineer"),
    (r"\bmachine learning\s+(?:engineer|developer)\b|\bml\s+engineer\b", "Machine Learning Engineer"),
    (r"\bai\s*[/&-]?\s*ml\b|artificial intelligence|\bai engineer\b", "AI / ML Engineer"),
    (r"\bdevops\b|site reliability|\bsre\b", "DevOps / SRE"),
    (r"\bcloud\s+(?:engineer|architect|consultant)\b", "Cloud Engineer"),
    (r"\bsoftware\b|\bdeveloper\b|\bprogrammer\b|application (?:lead|engineer)", "Software Development"),
    (r"\bproduct manager\b|\bproduct management\b", "Product Management"),
    (r"\bproject manager\b|\bprogram manager\b", "Project / Program Management"),
    (r"\bquality\b|\bqa\b|\btesting\b|\btester\b", "Quality / Testing"),
    (r"\bsales\b|business development|relationship manager", "Sales / Business Development"),
    (r"\bmarketing\b|\bseo\b|\bbrand manager\b", "Marketing"),
    (r"\brecruit|human resources|\bhr\b|talent acquisition", "HR / Recruitment"),
    (r"\baccountant\b|\bfinance\b|financial analyst|\btax\b|\baudit", "Finance / Accounting"),
    (r"customer (?:support|care|service)|\bbpo\b|voice process|call center", "Customer Support / BPO"),
    (r"\boperations?\b", "Operations"),
    (r"supply chain|\blogistics\b|\bprocurement\b|\bpurchase\b", "Supply Chain / Logistics"),
    (r"\bmechanical\b|maintenance engineer", "Mechanical / Maintenance"),
    (r"\belectrical\b|\belectronics\b", "Electrical / Electronics"),
    (r"\bcivil\b|construction|site engineer", "Civil / Construction"),
    (r"\bdoctor\b|\bnurse\b|\bmedical\b|\bpharma", "Healthcare / Pharma"),
    (r"\bteacher\b|\bfaculty\b|\btrainer\b|\bprofessor\b", "Education / Training"),
    (r"\bdesigner\b|\bgraphic\b|\bux\b|\bui\b|creative", "Design / Creative"),
    (r"\blegal\b|\blawyer\b|\bcounsel\b", "Legal"),
]

def normalize_job_role(title):
    text = "" if pd.isna(title) else str(title).casefold()
    return next((role for pattern, role in TITLE_RULES if re.search(pattern, text, flags=re.I)), "Other")

LOCATION_ALIASES = {
    "bangalore": "Bengaluru", "bengaluru": "Bengaluru", "gurgaon": "Gurugram",
    "gurugram": "Gurugram", "bombay": "Mumbai", "new delhi": "Delhi",
    "delhi ncr": "Delhi NCR", "remote": "Remote", "work from home": "Remote",
}

def normalize_location(value):
    if pd.isna(value):
        return "Not specified"
    text = re.sub(r"\([^)]*\)", "", str(value))
    text = re.sub(r"^hybrid\s*[-:]\s*", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip(" ,-\t")
    return LOCATION_ALIASES.get(text.casefold(), text.title()) if text else "Not specified"


## Skills extraction and normalization

In [6]:
SKILL_ALIASES = {
    "python3": "Python", "python 3": "Python", "python": "Python",
    "sql": "SQL", "structured query language": "SQL", "ms sql": "SQL",
    "powerbi": "Power BI", "power bi": "Power BI", "ms power bi": "Power BI",
    "tableau": "Tableau", "ms excel": "Excel", "microsoft excel": "Excel", "excel": "Excel",
    "postgres": "PostgreSQL", "postgresql": "PostgreSQL", "machine learning": "Machine Learning",
    "ml": "Machine Learning", "amazon web services": "AWS", "aws": "AWS",
    "microsoft azure": "Azure", "azure": "Azure", "google cloud platform": "GCP", "gcp": "GCP",
    "apache spark": "Spark", "spark": "Spark", "apache hadoop": "Hadoop", "hadoop": "Hadoop",
    "snowflake": "Snowflake", "dbt": "dbt", "etl": "ETL", "statistics": "Statistics",
    "statistical analysis": "Statistics", "deep learning": "Deep Learning",
    "natural language processing": "NLP", "nlp": "NLP", "numpy": "NumPy", "pandas": "Pandas",
    "docker": "Docker", "kubernetes": "Kubernetes", "git": "Git", "r programming": "R",
    "r language": "R", "devops": "DevOps", "azure devops": "Azure DevOps",
    "communication skills": "Communication",
}

CORE_PATTERNS = {
    "Python": r"(?<![\w])python(?:\s*3)?(?![\w])",
    "SQL": r"(?<![\w])sql(?![\w])|structured query language",
    "Excel": r"(?<![\w])(?:ms |microsoft )?excel(?![\w])",
    "Power BI": r"(?<![\w])power\s*bi(?![\w])", "Tableau": r"\btableau\b",
    "Machine Learning": r"\bmachine learning\b", "Pandas": r"\bpandas\b", "NumPy": r"\bnumpy\b",
    "AWS": r"\baws\b|amazon web services", "Azure": r"\bazure\b",
    "GCP": r"\bgcp\b|google cloud platform", "Spark": r"\b(?:apache )?spark\b",
    "Hadoop": r"\bhadoop\b", "Snowflake": r"\bsnowflake\b", "dbt": r"(?<![\w])dbt(?![\w])",
    "ETL": r"(?<![\w])etl(?![\w])", "Statistics": r"\bstatistic(?:s|al)\b",
    "Deep Learning": r"\bdeep learning\b", "NLP": r"(?<![\w])nlp(?![\w])|natural language processing",
    "Docker": r"\bdocker\b", "Kubernetes": r"\bkubernetes\b", "Git": r"(?<![\w])git(?![\w])",
}

def normalize_skill(value):
    if pd.isna(value):
        return None
    text = re.sub(r"\s+", " ", str(value)).strip(" .,-_/|")
    if len(text) < 2 or len(text) > 80:
        return None
    canonical = SKILL_ALIASES.get(text.casefold())
    return canonical or (text.upper() if len(text) <= 4 and text.isalpha() else text.title())

def extract_skills(skill_text, description=None):
    found = []
    if pd.notna(skill_text):
        for token in re.split(r"[,;|\n]+", str(skill_text)):
            normalized = normalize_skill(token)
            if normalized:
                found.append(normalized)
    combined = " ".join(str(value) for value in (skill_text, description) if pd.notna(value)).casefold()
    found.extend(skill for skill, pattern in CORE_PATTERNS.items() if re.search(pattern, combined, flags=re.I))
    return list(dict.fromkeys(found))

def skill_cooccurrence(skill_rows, top_n=20):
    top = set(skill_rows["skill"].value_counts().head(top_n).index)
    counter = Counter()
    for _, values in skill_rows[skill_rows["skill"].isin(top)].groupby("job_id")["skill"]:
        counter.update(combinations(sorted(set(values)), 2))
    return pd.DataFrame(
        [(left, right, count) for (left, right), count in counter.items()],
        columns=["skill_1", "skill_2", "job_count"],
    ).sort_values("job_count", ascending=False, ignore_index=True)

## Build clean job, skill, and location tables

In [7]:
jobs = raw.copy()
jobs.columns = [normalize_column_name(column) for column in jobs.columns]
required = {"title", "job_id", "company_name", "location"}
missing_required = required - set(jobs.columns)
if missing_required:
    raise ValueError(f"Required fields are missing: {sorted(missing_required)}")

raw_row_count = len(jobs)
exact_duplicates = int(jobs.duplicated().sum())
jobs = (
    jobs.assign(_completeness=jobs.notna().sum(axis=1))
    .sort_values(["job_id", "_completeness"], ascending=[True, False])
    .drop_duplicates("job_id", keep="first")
    .drop(columns="_completeness")
    .copy()
)

for column in ["title", "company_name", "currency", "salary", "experience", "location", "tags_and_skills"]:
    if column in jobs:
        jobs[column] = jobs[column].map(clean_text)
if "job_description" not in jobs:
    jobs["job_description"] = pd.NA
jobs["job_description"] = jobs["job_description"].map(clean_html_text)

jobs["original_job_title"] = jobs["title"]
jobs["normalized_job_role"] = jobs["title"].map(normalize_job_role)
jobs["original_location"] = jobs["location"]
location_lists = jobs["location"].map(
    lambda value: list(dict.fromkeys(normalize_location(token) for token in str(value).split(",")))
    if pd.notna(value) else ["Not specified"]
)
jobs["primary_location"] = location_lists.str[0]

salary_ranges = jobs["salary"].map(parse_salary)
jobs["minimum_salary"] = [value[0] for value in salary_ranges]
jobs["maximum_salary"] = [value[1] for value in salary_ranges]
jobs["average_salary"] = jobs[["minimum_salary", "maximum_salary"]].mean(axis=1)
is_inr = jobs.get("currency", pd.Series("INR", index=jobs.index)).eq("INR")
jobs["minimum_salary_lpa"] = jobs["minimum_salary"].where(is_inr) / 100_000
jobs["maximum_salary_lpa"] = jobs["maximum_salary"].where(is_inr) / 100_000
jobs["average_salary_lpa"] = jobs["average_salary"].where(is_inr) / 100_000

experience_ranges = jobs.get("experience", pd.Series(pd.NA, index=jobs.index)).map(parse_experience)
parsed_min = pd.Series([value[0] for value in experience_ranges], index=jobs.index)
parsed_max = pd.Series([value[1] for value in experience_ranges], index=jobs.index)
source_min = pd.to_numeric(jobs.get("minimum_experience"), errors="coerce")
source_max = pd.to_numeric(jobs.get("maximum_experience"), errors="coerce")
jobs["minimum_experience"] = source_min.combine_first(parsed_min)
jobs["maximum_experience"] = source_max.combine_first(parsed_max)
jobs["average_experience"] = jobs[["minimum_experience", "maximum_experience"]].mean(axis=1)
jobs["experience_category"] = [
    experience_category(minimum, maximum)
    for minimum, maximum in zip(jobs["minimum_experience"], jobs["maximum_experience"])
]

jobs["company_rating"] = pd.to_numeric(jobs.get("aggregate_rating"), errors="coerce")
jobs["company_rating"] = jobs["company_rating"].where(jobs["company_rating"].between(1, 5))
jobs["job_id"] = pd.to_numeric(jobs["job_id"], errors="coerce").astype("Int64")
jobs["skills_raw"] = jobs.get("tags_and_skills", pd.Series(pd.NA, index=jobs.index))

skill_records = []
for row in jobs[["job_id", "skills_raw", "job_description"]].itertuples(index=False):
    skill_records.extend((row.job_id, skill) for skill in extract_skills(row.skills_raw, row.job_description))
job_skills = pd.DataFrame(skill_records, columns=["job_id", "skill"]).drop_duplicates(ignore_index=True)

location_records = [
    (job_id, location)
    for job_id, values in zip(jobs["job_id"], location_lists)
    for location in values
]
job_locations = pd.DataFrame(location_records, columns=["job_id", "location"]).drop_duplicates(ignore_index=True)

print(f"Raw rows: {raw_row_count:,}")
print(f"Clean unique jobs: {len(jobs):,}")
print(f"Exact duplicate rows found: {exact_duplicates:,}")
print(f"Duplicate job-ID rows removed: {raw_row_count - len(jobs):,}")
print(f"Unique job-skill rows: {len(job_skills):,}")
print(f"Unique job-location rows: {len(job_locations):,}")

Raw rows: 97,929
Clean unique jobs: 97,679
Exact duplicate rows found: 247
Duplicate job-ID rows removed: 250
Unique job-skill rows: 774,307
Unique job-location rows: 131,209


## Save processed outputs

In [8]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
jobs.to_parquet(OUTPUT_DIR / "clean_jobs.parquet", index=False)
job_skills.to_parquet(OUTPUT_DIR / "job_skills.parquet", index=False)
job_locations.to_parquet(OUTPUT_DIR / "job_locations.parquet", index=False)

report = {
    "source_file": source_file,
    "source_table": source_table,
    "raw_rows": raw_row_count,
    "clean_rows": len(jobs),
    "exact_duplicates": exact_duplicates,
    "duplicate_job_ids_removed": raw_row_count - len(jobs),
    "usable_inr_salaries": int(jobs["average_salary_lpa"].notna().sum()),
    "jobs_with_skills": int(jobs["job_id"].isin(job_skills["job_id"]).sum()),
}
(OUTPUT_DIR / "processing_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
display(pd.Series(report, name="value").to_frame())
print("Saved to:", OUTPUT_DIR.resolve())

,value
source_file,indian-job-market-dataset-2025.xlsx
source_table,Sheet1
raw_rows,97929
clean_rows,97679
exact_duplicates,247
duplicate_job_ids_removed,250
usable_inr_salaries,33362
jobs_with_skills,97186


Saved to: /content/job_market_outputs


## Job role demand

In [9]:
top_roles = (
    jobs.groupby("normalized_job_role")["job_id"].nunique()
    .nlargest(20).rename("job_count").reset_index()
)
display(top_roles)
fig = px.bar(
    top_roles.sort_values("job_count"), x="job_count", y="normalized_job_role",
    orientation="h", title="Most in-demand role groups",
    labels={"job_count": "Unique listings", "normalized_job_role": "Role group"},
)
fig.show()

,normalized_job_role,job_count
0,Other,47269
1,Software Development,13649
2,Sales / Business Development,11462
3,Customer Support / BPO,3350
4,HR / Recruitment,2610
5,Quality / Testing,2575
6,Finance / Accounting,2473
7,Operations,1801
8,Marketing,1571
9,Design / Creative,1372


## Skills demand and combinations

In [10]:
top_skills = job_skills.groupby("skill")["job_id"].nunique().nlargest(25).rename("job_count").reset_index()
top_skills["share_of_skilled_jobs"] = top_skills["job_count"] / job_skills["job_id"].nunique()
display(top_skills.style.format({"share_of_skilled_jobs": "{:.1%}"}))
px.bar(
    top_skills.sort_values("job_count"), x="job_count", y="skill", orientation="h",
    title="Most requested skills", labels={"job_count": "Unique listings", "skill": "Skill"},
).show()

pairs = skill_cooccurrence(job_skills, top_n=25).head(25).copy()
pairs["combination"] = pairs["skill_1"] + " + " + pairs["skill_2"]
display(pairs)
px.bar(
    pairs.sort_values("job_count"), x="job_count", y="combination", orientation="h",
    title="Frequent skill combinations", labels={"job_count": "Listings", "combination": "Combination"},
).show()

,skill,job_count,share_of_skilled_jobs
0,Sales,9178,9.4%
1,Python,7558,7.8%
2,SQL,7171,7.4%
3,Project Management,5544,5.7%
4,Customer Service,5122,5.3%
5,SAP,5043,5.2%
6,Management,4849,5.0%
7,Azure,4547,4.7%
8,Excel,4488,4.6%
9,AWS,4431,4.6%


,skill_1,skill_2,job_count,combination
0,Python,SQL,2280,Python + SQL
1,Business Development,Sales,2110,Business Development + Sales
2,AWS,Azure,2037,AWS + Azure
3,AWS,Python,1708,AWS + Python
4,Azure,SQL,1514,Azure + SQL
5,Azure,Python,1507,Azure + Python
6,Marketing,Sales,1422,Marketing + Sales
7,AWS,SQL,1217,AWS + SQL
8,CSS,Javascript,1199,CSS + Javascript
9,AWS,Kubernetes,1107,AWS + Kubernetes


## Salary analysis

In [11]:
salary_jobs = jobs[jobs["average_salary_lpa"].gt(0) & jobs["average_salary_lpa"].notna()].copy()
display(salary_jobs["average_salary_lpa"].describe(percentiles=[.01, .25, .5, .75, .95, .99]).to_frame())

p99 = salary_jobs["average_salary_lpa"].quantile(.99)
px.histogram(
    salary_jobs[salary_jobs["average_salary_lpa"] <= p99],
    x="average_salary_lpa", nbins=50,
    title="Average annual salary in LPA (display capped at 99th percentile)",
    labels={"average_salary_lpa": "Average salary (LPA)"},
).show()

role_salary = salary_jobs.groupby("normalized_job_role").agg(
    median_salary_lpa=("average_salary_lpa", "median"),
    mean_salary_lpa=("average_salary_lpa", "mean"),
    sample_size=("job_id", "nunique"),
).reset_index()
role_salary = role_salary[role_salary["sample_size"] >= 30].sort_values("median_salary_lpa", ascending=False)
display(role_salary)
px.bar(
    role_salary.head(15).sort_values("median_salary_lpa"),
    x="median_salary_lpa", y="normalized_job_role", orientation="h",
    hover_data=["sample_size"], title="Median salary by role (minimum 30 disclosed salaries)",
).show()

,average_salary_lpa
count,33362.000000
mean,7.534050
std,10.241594
min,0.005000
1%,0.750000
25%,2.875000
50%,4.200000
75%,8.000000
95%,25.000000
99%,45.000000


,normalized_job_role,median_salary_lpa,mean_salary_lpa,sample_size
8,Data Scientist,21.3750,20.380208,48
0,AI / ML Engineer,21.2500,25.464796,49
7,Data Engineer,19.6250,17.842339,310
22,Product Management,16.0000,18.606410,39
10,DevOps / SRE,15.0000,16.354441,152
26,Software Development,13.0000,13.697272,1923
23,Project / Program Management,11.0000,14.099886,175
2,Business Analyst,10.5000,11.695230,87
6,Data Analyst,7.7500,10.922131,61
16,Legal,6.0000,8.009848,66


## Salary by experience, location, and skill

In [12]:
experience_salary = salary_jobs.groupby("minimum_experience").agg(
    median_salary_lpa=("average_salary_lpa", "median"), sample_size=("job_id", "nunique")
).reset_index()
experience_salary = experience_salary[experience_salary["sample_size"] >= 20]
px.scatter(
    experience_salary, x="minimum_experience", y="median_salary_lpa", size="sample_size",
    title="Median salary vs minimum experience",
    labels={"minimum_experience": "Minimum experience (years)", "median_salary_lpa": "Median salary (LPA)"},
).show()

location_salary = salary_jobs.groupby("primary_location").agg(
    median_salary_lpa=("average_salary_lpa", "median"), sample_size=("job_id", "nunique")
).reset_index()
location_salary = location_salary[location_salary["sample_size"] >= 30].nlargest(20, "median_salary_lpa")
display(location_salary)

skill_salary = job_skills.merge(salary_jobs[["job_id", "average_salary_lpa"]], on="job_id")
skill_salary = skill_salary.groupby("skill").agg(
    median_salary_lpa=("average_salary_lpa", "median"), sample_size=("job_id", "nunique")
).reset_index()
skill_salary = skill_salary[skill_salary["sample_size"] >= 30].nlargest(20, "median_salary_lpa")
display(skill_salary)

,primary_location,median_salary_lpa,sample_size
175,Dahej,9.6250,36
382,Karimnagar,7.2500,33
80,Bareilly,7.0000,40
114,Bhopal,5.7500,44
303,Hyderabad,5.5000,3176
683,Remote,5.5000,793
498,Manesar,5.3750,59
49,Aurangabad,5.2500,173
215,Dubai,5.1250,44
649,Pune,5.0000,2745


,skill,median_salary_lpa,sample_size
3959,Cardiology,52.500,33
6223,DM,47.500,41
16639,Oncology,46.250,30
12391,Internal Medicine,32.500,37
6231,DNB,30.000,103
19691,Radiology,28.725,54
10166,General Medicine,27.500,63
13453,LLM,27.500,40
19322,Pytorch,27.500,30
10176,General Surgery,26.250,33


## Location and company analysis

In [13]:
top_locations = jobs.groupby("primary_location")["job_id"].nunique().nlargest(20).rename("job_count").reset_index()
top_companies = jobs.groupby("company_name")["job_id"].nunique().nlargest(20).rename("job_count").reset_index()

display(top_locations)
px.bar(
    top_locations.sort_values("job_count"), x="job_count", y="primary_location", orientation="h",
    title="Top primary locations", labels={"job_count": "Unique listings"},
).show()

display(top_companies)
px.bar(
    top_companies.sort_values("job_count"), x="job_count", y="company_name", orientation="h",
    title="Companies with the most listings", labels={"job_count": "Unique listings"},
).show()

,primary_location,job_count
0,Bengaluru,19426
1,Hyderabad,12333
2,Pune,9378
3,Mumbai,6632
4,Chennai,6425
5,Gurugram,5259
6,Noida,4979
7,Ahmedabad,2436
8,Kolkata,1962
9,Remote,1952


,company_name,job_count
0,Accenture,8193
1,IDESLABS PRIVATE LIMITED,2294
2,Wipro,1494
3,Krazy Mantra HR Solutions Pvt. Ltd,1026
4,PRO Hr Complete Solutions,981
5,Kotak Mahindra Bank,980
6,Infosys,937
7,Capgemini,698
8,IBM,687
9,PRADEEPIT CONSULTING SERVICES PVT LTD,615


## Data Analyst focus

In [14]:
analyst_jobs = jobs[jobs["normalized_job_role"].eq("Data Analyst")].copy()
analyst_skills = job_skills[job_skills["job_id"].isin(analyst_jobs["job_id"])].copy()
core = ["SQL", "Python", "Excel", "Power BI", "Tableau", "Statistics", "ETL", "AWS", "Azure", "Snowflake"]
core_demand = (
    analyst_skills[analyst_skills["skill"].isin(core)]
    .groupby("skill")["job_id"].nunique().reindex(core, fill_value=0)
    .rename("job_count").reset_index()
)
core_demand["share_of_data_analyst_jobs"] = core_demand["job_count"] / max(len(analyst_jobs), 1)
print("Data Analyst listings:", len(analyst_jobs))
display(core_demand.style.format({"share_of_data_analyst_jobs": "{:.1%}"}))
px.bar(core_demand.sort_values("job_count"), x="job_count", y="skill", orientation="h", title="Core skill demand for Data Analyst listings").show()

print("Most common Data Analyst skill pairs")
display(skill_cooccurrence(analyst_skills, top_n=20).head(15))

analyst_entry = analyst_jobs[analyst_jobs["minimum_experience"].le(2)]
analyst_entry_skills = job_skills[job_skills["job_id"].isin(analyst_entry["job_id"])]
print("Entry-level Data Analyst listings:", len(analyst_entry))
display(analyst_entry_skills.groupby("skill")["job_id"].nunique().nlargest(15).rename("job_count").reset_index())

analyst_skill_salary = analyst_skills.merge(
    analyst_jobs[["job_id", "average_salary_lpa"]].dropna(), on="job_id"
).groupby("skill").agg(
    median_salary_lpa=("average_salary_lpa", "median"), sample_size=("job_id", "nunique")
).reset_index()
display(analyst_skill_salary[analyst_skill_salary["sample_size"] >= 10].nlargest(15, "median_salary_lpa"))

Data Analyst listings: 301


,skill,job_count,share_of_data_analyst_jobs
0,SQL,143,47.5%
1,Python,122,40.5%
2,Excel,83,27.6%
3,Power BI,103,34.2%
4,Tableau,78,25.9%
5,Statistics,70,23.3%
6,ETL,27,9.0%
7,AWS,12,4.0%
8,Azure,14,4.7%
9,Snowflake,18,6.0%


Most common Data Analyst skill pairs


,skill_1,skill_2,job_count
0,Python,SQL,88
1,Data Analysis,Python,76
2,Data Analysis,SQL,73
3,Power BI,SQL,70
4,SQL,Tableau,56
5,Power BI,Tableau,56
6,Power BI,Python,55
7,Data Analysis,Power BI,50
8,Excel,SQL,49
9,Excel,Power BI,47


Entry-level Data Analyst listings: 105


,skill,job_count
0,Data Analysis,55
1,SQL,38
2,Power BI,33
3,Excel,31
4,Python,29
5,Data Management,21
6,Statistics,21
7,Tableau,19
8,DATA,17
9,Data Analytics,16


,skill,median_salary_lpa,sample_size
176,SQL,13.6000,34
197,Tableau,10.0000,27
156,Python,10.0000,21
41,DATA,7.9375,12
192,Statistics,7.7500,11
47,Data Analysis,7.5000,29
81,Excel,6.5000,27
145,Power BI,6.2500,33
2,Advanced Excel,5.7500,12


## Fresher and entry-level analysis

In [15]:
strict_fresher = jobs[jobs["minimum_experience"].eq(0) & jobs["maximum_experience"].le(2)]
entry_level = jobs[jobs["minimum_experience"].le(2)]
fresher_skills = job_skills[job_skills["job_id"].isin(strict_fresher["job_id"])]

print(f"Strict fresher listings: {len(strict_fresher):,}")
print(f"Broader entry-level listings: {len(entry_level):,}")
print("Top fresher role groups")
display(strict_fresher.groupby("normalized_job_role")["job_id"].nunique().nlargest(15).rename("job_count").reset_index())
print("Top fresher skills")
display(fresher_skills.groupby("skill")["job_id"].nunique().nlargest(15).rename("job_count").reset_index())
print("Top fresher locations")
display(strict_fresher.groupby("primary_location")["job_id"].nunique().nlargest(15).rename("job_count").reset_index())
print("Top fresher companies")
display(strict_fresher.groupby("company_name")["job_id"].nunique().nlargest(15).rename("job_count").reset_index())

Strict fresher listings: 5,637
Broader entry-level listings: 40,465
Top fresher role groups


,normalized_job_role,job_count
0,Other,2836
1,Sales / Business Development,769
2,Customer Support / BPO,584
3,HR / Recruitment,291
4,Operations,167
5,Healthcare / Pharma,164
6,Marketing,151
7,Finance / Accounting,147
8,Software Development,147
9,Design / Creative,78


Top fresher skills


,skill,job_count
0,Communication,871
1,Customer Service,855
2,Sales,644
3,English,564
4,Voice Process,410
5,Market Research,393
6,Excel,379
7,Content Writing,372
8,Stress Analysis,345
9,Moderation,336


Top fresher locations


,primary_location,job_count
0,Bengaluru,694
1,Chennai,490
2,Hyderabad,431
3,Navi Mumbai,390
4,Pune,349
5,Noida,337
6,Gurugram,303
7,Mumbai,296
8,Ahmedabad,190
9,Kolkata,120


Top fresher companies


,company_name,job_count
0,Accenture,539
1,eClerx,117
2,Essentiallysports,66
3,Bajaj Finance,59
4,Teleperformance (TP),55
5,DECATHLON Sports,51
6,Resolve Medicode,51
7,Naukri Hospitality Jobs,49
8,Genpact,44
9,Takecare Manpower Services,38


## Key findings and limitations

In [16]:
summary = {
    "unique_jobs": int(jobs["job_id"].nunique()),
    "unique_companies": int(jobs["company_name"].nunique(dropna=True)),
    "unique_primary_locations": int(jobs["primary_location"].nunique(dropna=True)),
    "usable_inr_salary_rows": int(salary_jobs["job_id"].nunique()),
    "median_disclosed_inr_salary_lpa": float(salary_jobs["average_salary_lpa"].median()),
    "top_named_role": top_roles.loc[top_roles["normalized_job_role"].ne("Other"), "normalized_job_role"].iloc[0],
    "top_skill": top_skills.iloc[0]["skill"],
    "top_location": top_locations.iloc[0]["primary_location"],
    "top_company": top_companies.iloc[0]["company_name"],
    "strict_fresher_jobs": len(strict_fresher),
}
display(pd.Series(summary, name="value").to_frame())

print("Limitations:")
print("• One supplied source snapshot; results are not the complete Indian labor market.")
print("• Missing/unpaid salaries are excluded, and USD salaries are not converted.")
print("• Relative upload labels cannot support a reliable calendar trend.")
print("• Role and skill normalization simplify free text and should be reviewed for specialized use.")
print("• Salary and skill patterns are associations, not evidence of causation.")

,value
unique_jobs,97679
unique_companies,18616
unique_primary_locations,1298
usable_inr_salary_rows,33362
median_disclosed_inr_salary_lpa,4.2
top_named_role,Software Development
top_skill,Sales
top_location,Bengaluru
top_company,Accenture
strict_fresher_jobs,5637


Limitations:
• One supplied source snapshot; results are not the complete Indian labor market.
• Missing/unpaid salaries are excluded, and USD salaries are not converted.
• Relative upload labels cannot support a reliable calendar trend.
• Role and skill normalization simplify free text and should be reviewed for specialized use.
• Salary and skill patterns are associations, not evidence of causation.


## 17. Optional: download generated outputs from Colab

Run this cell when you want a ZIP archive containing the processed Parquet files and processing report.

In [17]:
if IN_COLAB:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/job_market_outputs", "zip", OUTPUT_DIR)
    print("Created:", archive)
    # Uncomment the next line to download immediately:
    # files.download(archive)
else:
    print("Processed outputs are available at:", OUTPUT_DIR.resolve())

Created: /content/job_market_outputs.zip
